# Multilingual Fake News Detector — Colab training

Fine-tune `bert-base-multilingual-cased` on six languages (English, Urdu,
Spanish, German, Chinese, Korean) on Colab, then download the exported
TensorFlow SavedModel to drop into `./saved_model/` locally.

**Before running:** Runtime > Change runtime type > Hardware accelerator = **GPU (T4)**.

Then just **Runtime > Run all**. Every cell is pre-filled — nothing to edit.

## 1. Get the project files

Clones the repo (already filled in) and moves into it. Safe to re-run.

In [7]:
import os

REPO_URL = "https://github.com/ammarali-ai/Fake-News-Detection.git"
if not os.path.isdir("/content/Fake-News-Detection"):
    !git clone $REPO_URL /content/Fake-News-Detection
%cd /content/Fake-News-Detection
!ls

Cloning into '/content/Fake-News-Detection'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 107 (delta 41), reused 93 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 84.18 KiB | 4.21 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/Fake-News-Detection
api.py		 data_prep.py	     docs	      README.md
app.py		 deploy.ps1	     evaluate.py      requirements.txt
CHANGELOG.md	 deploy.sh	     LICENSE	      tests
CONTRIBUTING.md  docker-compose.yml  model_loader.py  train.py
data		 Dockerfile	     notebooks


## 2. Install dependencies

Uses Colab's built-in TensorFlow / NumPy (downgrading them is what caused the
earlier NumPy-2 crash). We only add `transformers`, `datasets`, and `tf-keras`
(the Keras-2 shim the TF transformer models need on modern TensorFlow).

`TF_USE_LEGACY_KERAS=1` makes `tf.keras` use that shim. **No runtime restart needed.**

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

!pip install -q -U transformers datasets tf-keras sentencepiece scikit-learn openpyxl

import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 3. Build the corpus

Downloads the English/Urdu/Spanish datasets from the HuggingFace Hub and merges
them with the committed German/Chinese/Korean seed sets. `--subset 2000` caps
each language to 2000 rows so the classes stay balanced and training is quick;
drop it for the full corpus.

In [ ]:
!python data_prep.py --subset 2000

## 4. Train + export the SavedModel

Fine-tunes `bert-base-multilingual-cased` and writes `saved_model/`. A few
epochs over the balanced corpus take a handful of minutes on a T4 GPU.

In [ ]:
!TF_USE_LEGACY_KERAS=1 python train.py --csv data/train.csv --epochs 3 --batch-size 16

## 5. Evaluate (optional)

Prints accuracy / F1 overall and per language, and writes `evaluation_results.json`.

In [ ]:
!TF_USE_LEGACY_KERAS=1 python evaluate.py --csv data/test.csv

## 6. Download the SavedModel

Zips `saved_model/` and downloads `saved_model.zip` to your computer. Unzip it
into the repo root locally so `./saved_model/saved_model.pb` exists.

In [ ]:
import shutil
from google.colab import files

assert os.path.isdir("saved_model"), "saved_model/ not found — did cells 3 and 4 finish without errors?"
shutil.make_archive("saved_model", "zip", "saved_model")
files.download("saved_model.zip")